# payment_risk_v1 exploration

What this notebook does, top to bottom:

1. build (or load) a versioned `payment_risk_v1` training set through Feast and split it by time;
2. look at the data: fraud rate and volume per day, nulls, which features separate fraud;
3. test which derived request features earn their place;
4. compare five models, from a linear baseline to the GPU model the pipeline trains;
5. pick operating thresholds, check the held-out test, calibration and feature importance;
6. run the pipeline's own gate on the result and save the model.

Every data and training step is a function in `risk_model.steps`, shared with the `payment-risk-training` pipeline. Negatives are down-sampled, so every metric below uses the stored sample weights and reflects the real fraud rate.

In [ ]:
import json
import os
import tempfile
import uuid
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import precision_recall_curve

from risk_model import steps
from risk_model.data import DatasetReader
from risk_model.evaluate import DECLINE_FPR, STEP_UP_FPR, GateEvaluator
from risk_model.split import WindowSplitter
from risk_model.train import RiskTrainer, load_booster, score
from risk_model.windows import rolling

## 0. Windows and settings

`rolling` lays out three contiguous windows that end on `END_DAY`, the same rule the scheduled run uses. On dev only 2026-10-14 to 2026-10-21 is replayed: use `END_DAY = date(2026, 10, 20)` with 5, 1 and 1 days. Override the settings with `NOTEBOOK_END_DAY`, `NOTEBOOK_TRAIN_DAYS`, `NOTEBOOK_VALIDATION_DAYS` and `NOTEBOOK_TEST_DAYS`, and the negative sampling rates with `NOTEBOOK_COMPARE_RATE` and `NOTEBOOK_ABLATION_RATE`. For the small dev dataset, use 0.1 for both rates.

In [ ]:
END_DAY = date.fromisoformat(os.environ.get("NOTEBOOK_END_DAY", "2026-10-28"))
plan = rolling(
    END_DAY,
    train_days=int(os.environ.get("NOTEBOOK_TRAIN_DAYS", "56")),
    validation_days=int(os.environ.get("NOTEBOOK_VALIDATION_DAYS", "14")),
    test_days=int(os.environ.get("NOTEBOOK_TEST_DAYS", "14")),
)
windows = {
    "train": str(plan.train),
    "validation": str(plan.validation),
    "test": str(plan.test),
}
cutoff = plan.cutoff
RANDOM = 7
COMPARE_RATE = float(os.environ.get("NOTEBOOK_COMPARE_RATE", "0.05"))
ABLATION_RATE = float(os.environ.get("NOTEBOOK_ABLATION_RATE", "0.02"))
run_id = str(uuid.uuid4())
splitter = WindowSplitter(**windows)
windows, cutoff

## 1. Training set

Leave `NOTEBOOK_SNAPSHOT` unset to build a new version through Feast, or set it to a recorded snapshot ID to load it.

In [ ]:
snapshot = os.environ.get("NOTEBOOK_SNAPSHOT", "")
if not snapshot:
    planned = steps.plan_examples(**windows, run_id=run_id)
    for chunk in json.loads(planned["chunks"]):
        steps.stage_examples(chunk=chunk, cutoff=cutoff)
    snapshot = steps.assemble_examples(**windows, cutoff=cutoff, run_id=run_id)["snapshot"]
reader = DatasetReader.at(snapshot)
print(json.dumps(reader.provenance(), indent=2))

In [ ]:
print(steps.check_examples(snapshot=snapshot, run_id=run_id))
print(steps.split_windows(snapshot=snapshot, **windows, run_id=run_id))

## 2. The data

`COMPARE_RATE` keeps only a share of the stored negatives in the training split so the slower models in section 4 finish quickly. Validation and test keep the stored sample.

In [ ]:
splits = {
    "train": reader.read(splitter["train"], COMPARE_RATE),
    "validation": reader.read(splitter["validation"]),
    "test": reader.read(splitter["test"]),
}
train, validation, test = splits["train"], splits["validation"], splits["test"]
pd.DataFrame(
    {
        name: {
            "rows": len(split.y),
            "positives": int(split.y.sum()),
            "fraud rate (weighted)": float(np.average(split.y, weights=split.w)),
        }
        for name, split in splits.items()
    }
).T

In [ ]:
frame = reader.scan(columns=["started_at", "label"]).to_pandas()
frame["weight"] = np.where(frame["label"].astype(bool), 1.0, 1.0 / reader.sample_rate)
frame["fraud_weight"] = frame["weight"] * frame["label"].astype(float)
frame["day"] = pd.to_datetime(frame["started_at"]).dt.floor("D")
by_day = frame.groupby("day")[["fraud_weight", "weight"]].sum()
by_day["fraud rate"] = by_day["fraud_weight"] / by_day["weight"]

nulls = pd.Series(np.isnan(train.x).mean(axis=0), index=reader.features)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
by_day["fraud rate"].plot(ax=axes[0], marker="o")
axes[0].set_title("Fraud rate per day (weighted)")
by_day["weight"].plot(ax=axes[1], marker="o")
axes[1].set_title("Attempts per day (weighted)")
nulls.sort_values().tail(12).plot(kind="barh", ax=axes[2])
axes[2].set_title("Share of nulls, top 12 features")
plt.tight_layout()
plt.show()

In [ ]:
x = pd.DataFrame(train.x, columns=reader.features)
fraud = train.y > 0
gap = (x[fraud].mean() - x[~fraud].mean()) / x.std(ddof=0).replace(0.0, np.nan)
gap.abs().dropna().sort_values().tail(12).plot(kind="barh", figsize=(8, 4.5))
plt.title("Standardized mean gap, fraud vs legit (train sample)")
plt.show()

## 3. Derived request features

Each of the three derived candidates is removed in turn from a CPU model at `ABLATION_RATE` (2% of negatives by default). A candidate whose removal does not lower validation PR-AUC is proposed for exclusion. Changing the feature service requires coordinating its training and serving consumers; this notebook only reports the comparison.

In [ ]:
small = reader.read(splitter["train"], rate=ABLATION_RATE)
evaluator = GateEvaluator()


def pr_auc(names):
    """Returns the validation PR-AUC of a CPU model trained on the named features."""
    booster = RiskTrainer({"max_depth": 5}, device="cpu").fit(
        small.select(names), validation.select(names)
    )
    scores = {"model": score(booster, validation.select(names).x)}
    thresholds = {"model": evaluator.thresholds(validation.y, scores["model"], validation.w)}
    return evaluator.evaluate(thresholds, validation.y, validation.w, scores)["model"]["pr_auc"]


derived = [
    "request_features__customer_amount_to_30d_avg",
    "request_features__payment_method_amount_to_30d_avg",
    "request_features__scored_hour_utc",
]
full = pr_auc(reader.features)
ablation = {"all features": full}
for name in derived:
    ablation[f"without {name}"] = pr_auc([f for f in reader.features if f != name])
pd.Series(ablation, name="validation PR-AUC").round(4)

## 4. Five models, simple to strong

The first four train on this machine. Logistic regression is the same baseline the pipeline's gate uses, and the CPU XGBoost model uses the pipeline's trainer. Each model gets its own thresholds from validation: one at 0.1% false-positive rate (decline) and one at 1% (step-up), so the table compares them at the same operating cost.

In [ ]:
trainer = RiskTrainer({"max_depth": 5}, seed=RANDOM, device="cpu")
fitted = {
    "1. Logistic regression (baseline)": trainer.baseline(train),
    "2. Random forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=50,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM,
    ).fit(train.x, train.y, sample_weight=train.w),
    "3. Histogram gradient boosting": HistGradientBoostingClassifier(
        max_depth=5,
        learning_rate=0.05,
        max_iter=300,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=RANDOM,
    ).fit(train.x, train.y, sample_weight=train.w),
    "4. XGBoost (CPU)": trainer.fit(train, validation),
}


def scored(split):
    """Returns each fitted model's scores for a split."""
    return {name: score(model, split.x) for name, model in fitted.items()}


def metrics(split, scores, thresholds):
    """Returns a table of metrics per model on a split, best PR-AUC first."""
    table = pd.DataFrame(evaluator.evaluate(thresholds, split.y, split.w, scores)).T
    return table.sort_values("pr_auc", ascending=False)


def plot_comparison(table, scores, split, title):
    """Plots the metric bars and the precision-recall curves of every model."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
    table[["pr_auc", "recall_at_1pct_fpr"]].plot(kind="bar", ax=axes[0], ylim=(0, 1.05))
    axes[0].set_title(f"{title}: PR-AUC and recall at 1% FPR")
    plt.setp(axes[0].get_xticklabels(), rotation=20, ha="right")
    for name, values in scores.items():
        precision, recall, _ = precision_recall_curve(split.y, values, sample_weight=split.w)
        axes[1].plot(recall, precision, label=name)
    axes[1].set(xlabel="recall", ylabel="precision", title=f"{title}: precision-recall")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


validation_scores = scored(validation)
thresholds = {
    name: evaluator.thresholds(validation.y, values, validation.w)
    for name, values in validation_scores.items()
}
validation_table = metrics(validation, validation_scores, thresholds)
plot_comparison(validation_table, validation_scores, validation, "Validation")
validation_table.round(4)

## 5. The pipeline's model on the GPU

The same TrainJob as the pipeline's `fit_on_gpu` step, on the full stored sample. Notebooks cannot take GPU claims, so training runs there. It joins the comparison as model 5. It sees every stored negative, while models 1 to 4 use `COMPARE_RATE`, so expect it to lead.

In [ ]:
fit = steps.fit_on_gpu(snapshot=snapshot, **windows, params="{}", run_id=run_id)
with tempfile.TemporaryDirectory() as scratch:
    model_file = steps.download(
        fit["model_uri"].rstrip("/") + "/model.json", Path(scratch) / "model.json"
    )
    booster, order = load_booster(model_file)
assert order == reader.features
pipeline_model = "5. XGBoost (GPU, pipeline)"
fitted[pipeline_model] = booster

validation_scores = scored(validation)
thresholds = {
    name: evaluator.thresholds(validation.y, values, validation.w)
    for name, values in validation_scores.items()
}
validation_table = metrics(validation, validation_scores, thresholds)
plot_comparison(validation_table, validation_scores, validation, "Validation")
validation_table.round(4)

## 6. Operating thresholds

The decline threshold sits where 0.1% of legitimate attempts would be declined, and step-up where 1% would. These are the score levels the fraud service can act on.

In [ ]:
precision, recall, cuts = precision_recall_curve(
    validation.y, validation_scores[pipeline_model], sample_weight=validation.w
)
plt.figure(figsize=(8, 4.5))
plt.plot(cuts, precision[:-1], label="precision")
plt.plot(cuts, recall[:-1], label="recall")
for label, value in thresholds[pipeline_model].items():
    plt.axvline(value, ls="--", c="k")
    plt.annotate(f"{label} {value:.3f}", (value, 0.5), rotation=90, va="center", ha="right")
plt.xlabel("score")
plt.legend()
plt.title(f"Validation operating points ({DECLINE_FPR:.1%} and {STEP_UP_FPR:.0%} FPR)")
plt.show()
thresholds[pipeline_model]

## 7. Held-out test

Thresholds come from validation and are applied unchanged to the test window. Lift is the share of fraud caught when only the 5% highest-scoring attempts are reviewed.

In [ ]:
def lift(split, values, share=0.05):
    """Returns the share of fraud caught by reviewing the top `share` of scores."""
    order = np.argsort(-values)
    weights, labels = split.w[order], split.y[order] > 0
    reviewed = np.searchsorted(np.cumsum(weights), share * weights.sum())
    return float(weights[:reviewed][labels[:reviewed]].sum() / weights[labels].sum())


test_scores = scored(test)
test_table = metrics(test, test_scores, thresholds)
test_table["recall_in_top_5pct"] = pd.Series(
    {name: lift(test, values) for name, values in test_scores.items()}
)
plot_comparison(test_table, test_scores, test, "Test")
test_table.round(4)

In [ ]:
flag = test_scores[pipeline_model] >= thresholds[pipeline_model]["decline"]
positive = test.y > 0
counts = np.array(
    [
        [test.w[~flag & ~positive].sum(), test.w[flag & ~positive].sum()],
        [test.w[~flag & positive].sum(), test.w[flag & positive].sum()],
    ]
)
plt.figure(figsize=(5, 4))
plt.imshow(counts, cmap="Blues")
for (row, column), value in np.ndenumerate(counts):
    plt.text(column, row, f"{value:,.0f}", ha="center", va="center")
plt.xticks([0, 1], ["approve", "decline"])
plt.yticks([0, 1], ["legit", "fraud"])
plt.xlabel("model decision")
plt.ylabel("actual")
plt.title("Test, weighted counts at the decline threshold")
plt.show()

## 8. What the model uses, and how well its scores are calibrated

Gain says which features the trees split on. Calibration compares the mean score in each decile of test scores with the weighted fraud rate in that decile; points near the diagonal mean a score can be read as a probability, which matters when a fixed threshold is enforced.

In [ ]:
gain = pd.Series(booster.get_score(importance_type="gain"))
gain.index = [reader.features[int(key[1:])] for key in gain.index]

frame = pd.DataFrame(
    {"score": test_scores[pipeline_model], "weight": test.w, "fraud": test.w * test.y}
)
frame["decile"] = pd.qcut(frame["score"], 10, duplicates="drop")
by_decile = frame.groupby("decile", observed=True).agg(
    predicted=("score", "mean"), fraud=("fraud", "sum"), weight=("weight", "sum")
)
by_decile["observed"] = by_decile["fraud"] / by_decile["weight"]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
gain.sort_values().tail(15).plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 15 features by gain (pipeline model)")
axes[1].plot(by_decile["predicted"], by_decile["observed"], marker="o")
axes[1].plot([0, by_decile["predicted"].max()], [0, by_decile["predicted"].max()], ls="--", c="k")
axes[1].set(xlabel="mean score", ylabel="weighted fraud rate", title="Calibration by score decile")
plt.tight_layout()
plt.show()

## 9. The pipeline's own gate, and saving

`steps.gate` is the step the pipeline runs: the model against the linear baseline and, once one exists, the production version, on the test window.

In [ ]:
verdict = steps.gate(snapshot=snapshot, **windows, model_uri=fit["model_uri"], run_id=run_id)
summary = json.loads(verdict["summary"])
print("gate passed:", verdict["passed"], summary["reasons"])
pd.DataFrame(summary["metrics"]).T.round(4)

In [ ]:
model_dir = Path(os.path.expanduser("~/models/payment_risk_v1"))
model_dir.mkdir(parents=True, exist_ok=True)
booster.save_model(model_dir / "model.json")

## 10. Summary

In [ ]:
best = validation_table["pr_auc"].idxmax()
model = test_table.loc[pipeline_model]
baseline = test_table.loc["1. Logistic regression (baseline)"]
Markdown(
    f"""
- **Best on validation:** {best}.
- **Pipeline model on test:** PR-AUC {model["pr_auc"]:.3f} against {baseline["pr_auc"]:.3f} for the linear baseline, recall at 1% FPR {model["recall_at_1pct_fpr"]:.3f} against {baseline["recall_at_1pct_fpr"]:.3f}.
- **Review lift:** reviewing the top 5% of scores catches {model["recall_in_top_5pct"]:.0%} of fraud, {model["recall_in_top_5pct"] / 0.05:.1f}x a random review.
- **Thresholds:** decline at score {thresholds[pipeline_model]["decline"]:.3f} (0.1% FPR on validation), step-up at {thresholds[pipeline_model]["step_up"]:.3f} (1% FPR). Start the fraud service's model threshold from the decline value, and revisit it on a larger sample.
- **Gate:** {"passed" if verdict["passed"] else "did not pass"} on this window.
"""
)